<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2011%20-%202026-05-07%20-%20Hip%C3%B3tesis%20y%20pruebas%20estad%C3%ADsticas/clase_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 11 · Hipótesis y pruebas estadísticas · 🧪 Taller

**Fecha:** Jueves 7 de mayo de 2026 · **Duración estimada:** ~60 min · **Nivel:** Intermedio

> 📖 **La teoría completa vive en el [HTML de la sesión](./index.html).** Este cuaderno es el **taller práctico**: aquí codificas.

## 🎯 Objetivos de aprendizaje

Al terminar este cuaderno serás capaz de:

1. **Formular** hipótesis estadísticas testables (H₀ y H₁).
2. **Ejecutar** t-test, chi-cuadrado y ANOVA con scipy.
3. **Interpretar** p-valores correctamente sin sobreestimar su poder.
4. **Comparar** tamaños de efecto (Cohen's d, Cramer's V) junto al p-valor.
5. **Diseñar** un protocolo de prueba para tu caso de estudio.

## ✅ Prerrequisitos

- Conocer distribuciones normales, media, desviación estándar (Día 8).
- Haber visto Día 10: visualizaciones (para gráficos de distribuciones).
- Pandas: `groupby()`, `loc[]`, `crosstab()`.

## 🧭 Cómo usar este cuaderno

- Ejecuta cada celda con `Shift + Enter`.
- Las celdas 🟢🟡🔴 son ejercicios (verde → rojo = dificultad).
- `# %% [solution]` = solución oculta. Descomenta si estás atascado.
- `assert` valida tu código: si no falla, está correcto ✓.

---

# Setup

In [ ]:
# --- Setup del entorno ---------------------------------------------------
# Imports estándar, reproducibilidad y confirmación visual
# -----------------------------------------------------------------------

# Stdlib
import random

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Estilo visual
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

print("Setup completo ✓")
print(f"pandas {pd.__version__} · scipy {stats.__version__}")

---

## 🎬 Hook · ¿Las mujeres del Titanic eran mayores que los hombres?

> **Pregunta de hoy:** Intuición dice que "primero las mujeres y los niños". ¿Fue cierto? ¿Hay diferencia estadística en la edad según sexo? ¿Cómo lo probamos sin simplemente "ver" un gráfico?

Carguemos los datos y veamos.

In [ ]:
# Cargar datos del Titanic
titanic = sns.load_dataset("titanic").dropna(subset=["age"])

print(f"Shape: {titanic.shape}")
print(f"\nEdad por sexo:")
print(titanic.groupby("sex")["age"].describe())

# Visualizar
plt.figure(figsize=(10, 4))
sns.violinplot(data=titanic, x="sex", y="age", palette="Set2")
plt.title("Distribución de edades por sexo (Titanic)")
plt.ylabel("Edad (años)")
plt.tight_layout()

⏸️ **Pause and think:** Mirando el gráfico, ¿dirías que las dos distribuciones son iguales? ¿O hay diferencia "obvia"? Pero... ¿qué es lo suficientemente diferente para llamarlo significativo?

---

## 1. t-test · Comparar 2 medias

⏱️ ~10 min

> 📌 **Concepto clave:** Un t-test compara la media de dos grupos y calcula si esa diferencia es "demasiado grande para ser azar". Devuelve un t-statístic y un p-valor. Si p < 0.05 (convencionalmente), rechazamos la H₀ ("no hay diferencia").

In [ ]:
# Demo: t-test para edad entre sexos

# Separar edades por sexo
edad_hombres = titanic.loc[titanic["sex"] == "male", "age"]
edad_mujeres = titanic.loc[titanic["sex"] == "female", "age"]

print(f"Hombres: n={len(edad_hombres)}, media={edad_hombres.mean():.2f}")
print(f"Mujeres: n={len(edad_mujeres)}, media={edad_mujeres.mean():.2f}")
print(f"Diferencia: {abs(edad_hombres.mean() - edad_mujeres.mean()):.2f} años")

# t-test de Student (Welch si varianzas distintas)
t_stat, p_value = stats.ttest_ind(edad_hombres, edad_mujeres, equal_var=False)

print(f"\nt-statístic = {t_stat:.3f}")
print(f"p-valor = {p_value:.4f}")
print(f"\nInterpretación:")
if p_value < 0.05:
    print("✓ Rechazamos H₀: SÍ hay diferencia significativa en edad entre sexos.")
else:
    print("✗ No rechazamos H₀: No hay evidencia de diferencia significativa.")

💡 **Tip:** `equal_var=False` usa la corrección de Welch. Es más seguro si sospechas que las varianzas son distintas.

### 🟢 Ejercicio 1 — Guiado · t-test entre grupos

⏱️ ~5 min

Usa el dataset `tips` (mesadas) y compara la **propina promedio** (`tip`) entre meseros (`sex="Male"`) y meseras (`sex="Female"`). Ejecuta un t-test e interpreta el resultado.

**Output esperado:** t-statístic, p-valor, conclusión ("significativo" o "no significativo").

In [ ]:
# 🟢 Completa el código

tips = sns.load_dataset("tips")

# Separar propinas por sexo
propina_male = tips.loc[tips["sex"] == "___", "tip"]  # ← completa
propina_female = tips.loc[tips["sex"] == "___", "tip"]  # ← completa

# t-test
t_stat, p_val = stats.ttest_ind(propina_male, propina_female, equal_var=False)

print(f"Propina promedio - Meseros: {propina_male.mean():.2f}, Meseras: {propina_female.mean():.2f}")
print(f"t = {t_stat:.3f}, p = {p_val:.4f}")
print(f"Conclusión: ", end="")
if p_val < 0.05:
    print("Diferencia significativa ✓")
else:
    print("Sin diferencia significativa ✗")

In [ ]:
# Validación
assert len(propina_male) > 0 and len(propina_female) > 0, "Valores vacíos"
assert isinstance(t_stat, (float, np.floating)), "t_stat debe ser número"
print("✅ Ejercicio 1 completado")

In [ ]:
# %% [solution]
# propina_male = tips.loc[tips["sex"] == "Male", "tip"]
# propina_female = tips.loc[tips["sex"] == "Female", "tip"]
# t_stat, p_val = stats.ttest_ind(propina_male, propina_female, equal_var=False)

---

## 2. Chi-cuadrado · Asociación entre categóricas

⏱️ ~10 min

> 📌 **Concepto clave:** El chi-cuadrado test mide si dos variables categóricas están asociadas o son independientes. Compara frecuencias **observadas** vs **esperadas bajo independencia**.

In [ ]:
# Demo: ¿Sexo y supervivencia están asociados en el Titanic?

# Crear tabla de contingencia
contingency_table = pd.crosstab(titanic["sex"], titanic["survived"])
print("Tabla de contingencia:")
print(contingency_table)
print()

# Chi-cuadrado
chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)

print(f"chi² = {chi2:.2f}")
print(f"p-valor = {p_val:.4e}")
print(f"grados de libertad = {dof}")
print(f"\nFrecuencias esperadas (bajo independencia):")
print(pd.DataFrame(expected, index=contingency_table.index, columns=contingency_table.columns))

if p_val < 0.05:
    print("\n✓ Sexo y supervivencia ESTÁN asociados (dependencia).")
else:
    print("\n✗ No hay evidencia de asociación.")

### 🟡 Ejercicio 2 — Aplicado · Chi-cuadrado con datos nuevos

⏱️ ~8 min

Usa el dataset `tips` e investiga si **sexo del cliente** y **si pasó o no el tiempo en fin de semana** (`time`) están asociados.

Pasos:
1. Crea un `crosstab(tips["sex"], tips["time"])`.
2. Ejecuta `chi2_contingency()`.
3. Interpreta: ¿asociación significativa?

**Output esperado:** chi², p-valor, conclusión.

In [ ]:
# 🟡 Completa el análisis

# Crear tabla
tab = pd.crosstab(tips["___"], tips["___"])  # ← rellena
print(tab)
print()

# Chi-cuadrado
chi2, p_val, dof, exp = stats.chi2_contingency(tab)

print(f"chi² = {chi2:.2f}, p = {p_val:.4f}")
print(f"Conclusión: ", end="")
if p_val < 0.05:
    print("Asociación significativa ✓")
else:
    print("Sin asociación ✗")

In [ ]:
# Validación
assert isinstance(chi2, (float, np.floating))
assert isinstance(p_val, (float, np.floating))
print("✅ Ejercicio 2 completado")

In [ ]:
# %% [solution]
# tab = pd.crosstab(tips["sex"], tips["time"])
# chi2, p_val, dof, exp = stats.chi2_contingency(tab)

---

## 3. ANOVA · Comparar 3 o más medias

⏱️ ~10 min

> 📌 **Concepto clave:** ANOVA (Analysis of Variance) extiende el t-test a 3+ grupos. Compara la variabilidad **entre grupos** con la variabilidad **dentro de grupos**. Si hay mucha más variación entre que dentro, hay efecto.

> ⚠️ **Advertencia:** No hacer múltiples t-tests entre cada par de grupos (inflación de error tipo I). ANOVA primero, luego test post-hoc si es significativo.

In [ ]:
# Demo: ANOVA — ¿La tarifa varía por clase?

# Separar tarifas por clase
tarifa_clase1 = titanic.loc[titanic["pclass"] == 1, "fare"].dropna()
tarifa_clase2 = titanic.loc[titanic["pclass"] == 2, "fare"].dropna()
tarifa_clase3 = titanic.loc[titanic["pclass"] == 3, "fare"].dropna()

print(f"Clase 1: media={tarifa_clase1.mean():.2f}, n={len(tarifa_clase1)}")
print(f"Clase 2: media={tarifa_clase2.mean():.2f}, n={len(tarifa_clase2)}")
print(f"Clase 3: media={tarifa_clase3.mean():.2f}, n={len(tarifa_clase3)}")

# ANOVA
f_stat, p_val = stats.f_oneway(tarifa_clase1, tarifa_clase2, tarifa_clase3)

print(f"\nF-statístic = {f_stat:.2f}")
print(f"p-valor = {p_val:.4e}")

if p_val < 0.05:
    print("\n✓ Hay diferencia significativa en tarifa entre clases.")
else:
    print("\n✗ Sin diferencia significativa.")

### 🟡 Ejercicio 3 — Aplicado · ANOVA con tips

⏱️ ~8 min

Compara el **total facturado** (`total_bill`) entre los **3 días** (Thurs, Fri, Sat, Sun). Usa ANOVA e interpreta.

Nota: si es significativo, comenta sobre cuál día tiene mayor gasto promedio.

In [ ]:
# 🟡 ANOVA con days

bill_thurs = tips.loc[tips["day"] == "___", "total_bill"]  # ← completa
bill_fri = tips.loc[tips["day"] == "___", "total_bill"]  # ← completa
bill_sat = tips.loc[tips["day"] == "___", "total_bill"]  # ← completa
bill_sun = tips.loc[tips["day"] == "___", "total_bill"]  # ← completa

# ANOVA
f_stat, p_val = stats.f_oneway(bill_thurs, bill_fri, bill_sat, bill_sun)

print(f"F = {f_stat:.2f}, p = {p_val:.4f}")
print("Conclusión: ", end="")
if p_val < 0.05:
    print("Diferencia significativa en gasto por día ✓")
else:
    print("Sin diferencia ✗")

# Resumen
print(f"\nPromedio por día:")
print(tips.groupby("day")["total_bill"].mean().sort_values(ascending=False))

In [ ]:
# Validación
assert len(bill_thurs) > 0 and len(bill_fri) > 0
print("✅ Ejercicio 3 completado")

In [ ]:
# %% [solution]
# bill_thurs = tips.loc[tips["day"] == "Thurs", "total_bill"]
# bill_fri = tips.loc[tips["day"] == "Fri", "total_bill"]
# bill_sat = tips.loc[tips["day"] == "Sat", "total_bill"]
# bill_sun = tips.loc[tips["day"] == "Sun", "total_bill"]

---

## 4. Tamaño del efecto · Cohen's d

⏱️ ~5 min

> 📌 **Concepto clave:** El p-valor nos dice "¿es la diferencia real?", pero no "¿qué tan grande es?". El tamaño del efecto lo hace. Cohen's d es la diferencia de medias dividida por la desviación estándar combinada.

> **Interpretación de Cohen's d:**
> - 0.2 = pequeño
> - 0.5 = mediano
> - 0.8 = grande

In [ ]:
# Helper: función para Cohen's d

def cohens_d(x, y):
    """
    Calcula Cohen's d para dos muestras.
    """
    nx, ny = len(x), len(y)
    dof = nx + ny - 2
    # Varianza combinada (pooled)
    s = np.sqrt(((nx - 1) * x.var() + (ny - 1) * y.var()) / dof)
    # d = diferencia de medias / desv combinada
    d = (x.mean() - y.mean()) / s
    return d

# Demo: Cohen's d para edad hombres vs mujeres
d = cohens_d(edad_hombres, edad_mujeres)
print(f"Cohen's d (edad: hombres vs mujeres) = {d:.3f}")
print(f"Interpretación: ", end="")
if abs(d) < 0.2:
    print("insignificante")
elif abs(d) < 0.5:
    print("pequeño")
elif abs(d) < 0.8:
    print("mediano")
else:
    print("grande")

### 🟢 Ejercicio 4 — Guiado · Cohen's d

⏱️ ~3 min

Calcula Cohen's d para la comparación de propinas (meseros vs meseras) que hiciste en el Ejercicio 1. ¿Es el efecto pequeño, mediano o grande?

In [ ]:
# 🟢 Calcula Cohen's d

d_tips = cohens_d(propina_male, propina_female)
print(f"Cohen's d = {d_tips:.3f}")
print(f"Magnitud: ", end="")
if abs(d_tips) < 0.2:
    print("insignificante")
elif abs(d_tips) < 0.5:
    print("pequeño")
elif abs(d_tips) < 0.8:
    print("mediano")
else:
    print("grande")

In [ ]:
assert isinstance(d_tips, (float, np.floating))
print("✅ Ejercicio 4 completado")

In [ ]:
# %% [solution]
# d_tips = cohens_d(propina_male, propina_female)

---

## 🔴 Ejercicio 5 — Desafío · Protocolo de hipótesis completo

⏱️ ~15-20 min · Open-ended

Diseña y ejecuta **un protocolo completo de prueba de hipótesis** sobre el dataset que prefieras (Titanic, tips, diamonds, penguins). El protocolo debe incluir:

1. **Formulación clara** de H₀ y H₁ (en palabras).
2. **Selección justificada del test** (¿por qué t-test, chi-cuadrado, ANOVA, etc.?).
3. **Supuestos verificados** (normalidad, homogeneidad de varianzas, si aplica).
4. **Ejecución del test** y reporte de p-valor.
5. **Tamaño del efecto** (Cohen's d, Cramer's V, o el apropiado).
6. **Conclusión en lenguaje de negocio** (no estadístico): qué significa el resultado para tu caso.

**Criterios de aceptación:**
- ✓ Hipótesis bien formuladas (no ambiguas)
- ✓ Test correcto para el tipo de datos
- ✓ Interpretación correcta del p-valor (no confundirlo con P(H₀))
- ✓ Mencionas el tamaño del efecto
- ✓ Conclusión en prosa (no solo números)

In [ ]:
# 🔴 Protocolo de hipótesis — Tu turno

# === SECCIÓN 1: FORMULACIÓN ===
# H₀: [escribe aquí en palabras claras]
# H₁: [escribe aquí]
# 
# Justificación: [por qué esperas rechazar o no rechazar H₀]

# === SECCIÓN 2: PREPARACIÓN DE DATOS ===
# [Carga el dataset, limpia si es necesario]

# ← Código aquí

# === SECCIÓN 3: VERIFICACIÓN DE SUPUESTOS ===
# [Si usas t-test: normalidad (gráfico Q-Q o Shapiro)]
# [Si usas ANOVA: igualdad de varianzas (Levene)]

# ← Código aquí

# === SECCIÓN 4: EJECUCIÓN DEL TEST ===
# [Ejecuta el test apropiado y captura p-valor]

# ← Código aquí

# === SECCIÓN 5: TAMAÑO DEL EFECTO ===
# [Calcula Cohen's d, Cramer's V, R², lo que corresponda]

# ← Código aquí

# === SECCIÓN 6: CONCLUSIÓN ===
# [En 3-4 oraciones, qué significa tu resultado para tu caso/negocio]

# ← Escribe aquí

print("\n🎯 Ejercicio 5: Entregado (revisión manual recomendada)")

---

## 🧭 Checkpoint · Auto-evaluación

Responde mentalmente:

1. ¿Qué significa un p-valor de 0.03?
   - a) Hay un 3% de chance de que H₀ sea cierta ← ✗ (error común)
   - b) Hay 3% de chance de observar estos datos si H₀ fuera verdad ← ✓
   - c) El efecto es pequeño ← ✗

2. ¿Cuándo usarías chi-cuadrado vs t-test?
   - a) Chi² para variables categóricas, t para numéricas ← ✓
   - b) Siempre chi² porque es más poderoso ← ✗
   - c) Depende del tamaño de muestra ← parcial

3. ¿Por qué "p-hacking" es malo?
   - a) Porque hace que el error tipo I explote (falsos positivos) ← ✓
   - b) Porque es ilegal ← ✗
   - c) Porque usa demasiada memoria ← ✗

📊 Has completado **5 secciones + 1 desafío**. ¡Dominas tests estadísticos!

In [ ]:
# Validación checkpoint
assert "t_stat" in dir() and "chi2" in dir() and "f_stat" in dir(), "Completa los ejercicios"
print("Checkpoint ✓ — dominas tests estadísticos")

---

## 🧠 Mental model de hoy

```
Pregunta de negocio
       ↓
  Formular H₀ y H₁
       ↓
¿Qué tipo de datos?
   ↙     ↓     ↘
2 num  3+ num  2 cat
   ↓     ↓      ↓
 t-test ANOVA Chi²
   ↓     ↓      ↓
 p-val Cohen's d / Cramer's V
   ↓     ↓      ↓
[Rechazar o no rechazar H₀]
       ↓
 Conclusión en palabras
```

## 📌 Resumen · Lo que aprendiste hoy

- **t-test:** Compara medias de 2 grupos. Reporta t-statístic y p-valor.
- **Chi-cuadrado:** Prueba independencia entre dos categóricas.
- **ANOVA:** Extiende t-test a 3+ grupos.
- **p-valor:** NO es "probabilidad de que H₀ sea cierta". Es probabilidad de datos tan extremos asumiendo H₀.
- **Tamaño del efecto:** Tan importante como el p-valor. Cohen's d para medias.
- **Supuestos:** Verifica normalidad, homogeneidad según el test.

## 🚀 Próximos pasos (D+1)

Mañana veremos **correlaciones y regresión lineal**. Para preparar:
- Relee la sección de "Errores tipo I y II" en el HTML si quieres fortalecer conceptos.
- Practica interpretando p-valores (no confundirlos con P(H₀)).

## 📚 Para profundizar

- [ASA Statement on p-values (2016)](https://www.amstat.org/asa/files/pdfs/P-ValueStatement.pdf) — lectura oficial
- [scipy.stats — documentación de tests](https://docs.scipy.org/doc/scipy/reference/stats.html)
- [Blog: The Problem with p-values](https://www.nature.com/articles/nature.2014.14716) — Nature, 2014

## ✍️ Reflexión final

> Explica en 3 oraciones por qué el p-valor de 0.05 es un umbral **arbitrario** pero útil. ¿Qué pasa si cambias a 0.01 o 0.10?

---

## 🧑‍🤝‍🧑 Trabajo en grupo — avance del proyecto

- **Hito 1:** Formulen 2 hipótesis sobre vuestro dataset del proyecto (ej: "ingresos varía por región").
- **Hito 2:** Ejecuten el test apropiado (t-test, chi², ANOVA).
- **Hito 3:** Reporten p-valor + tamaño del efecto.
- **Entrega:** Notebook `hypothesis_tests.ipynb` con los 2 tests + conclusiones en prosa (5-7 oraciones c/u).